# 8.3 Интерактив в Plotly — код страницы

Аналитика 101 · блок «Дата аналитика» · тема 08 «Визуализация в Python» · страница 8.3.

В первых четырёх примерах вы построите график выручки пяти каналов, настроите подсказку, создадите анимацию по месяцам и сохраните график в нескольких форматах. Ещё три примера показывают, как меняется размер HTML с числом точек, как подписать суммы на изображении и как добавить к графику таблицу для чтения на телефоне.

Ячейки с методом `show()` показывают готовые графики в ноутбуке. В первой из них попробуйте найти выручку приложения за июль 2025 года.

**Как работать.** Выполняйте ячейки сверху вниз. Для сохранения PNG и SVG пакету Kaleido понадобится Chrome или Chromium. Установить совместимый Chrome можно командой `plotly_get_chrome`.

Вывод получен на Python 3.13, NumPy 2.5, pandas 3.0, plotly 7.1 и kaleido 1.5.

## Где лежат данные

Распакуйте общий набор данных темы так, чтобы каталог `data/viz/` находился рядом с папкой урока. Первая ячейка найдёт каталог с файлом `sales_monthly.csv` и выберет рабочую папку.

В рабочей папке появятся девять файлов — `channels.html` (с библиотекой, почти 5 МБ), `channels_cdn.html`, `channels.json`, `channels.png`, `channels.svg`, `points.html` (миллион точек, 26 МБ), `plain.svg`, `titled.svg` и `mobile.html`. После работы их можно удалить.

In [1]:
import os
import pathlib
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
import plotly

root = pathlib.Path.cwd()
while not (root / "data" / "viz").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/viz/
assert (root / "data" / "viz" / "sales_monthly.csv").exists(), \
    "Не найдена папка data/viz/. Скачайте общий набор темы 08 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| plotly", plotly.__version__, "| kaleido", version("kaleido"))

Python 3.13.13 | numpy 2.5.3 | pandas 3.0.5 | plotly 7.1.0 | kaleido 1.5.0


## От таблицы до готового графика

### Каждому из пяти каналов соответствует отдельная линия

In [2]:
import pandas as pd
import plotly.express as px

sales = pd.read_csv("data/viz/sales_monthly.csv")  # длинный формат: строка = месяц × канал × страна
top5 = ["сайт", "маркетплейс", "приложение", "поиск", "рассылка"]
by_ch = (sales[sales["channel"].isin(top5)]
         .groupby(["month", "channel"], as_index=False)["revenue"].sum()
         .assign(mln=lambda d: d["revenue"] / 1e6))
print(by_ch.shape)
print(by_ch.head(3).to_string(index=False))

fig = px.line(by_ch, x="month", y="mln", color="channel",   # столбец → роль на графике
              category_orders={"channel": top5},             # порядок в легенде
              labels={"month": "месяц", "mln": "млн ₽", "channel": "канал"},
              title="С мая по июль 2025 года выручка приложения почти удвоилась")
fig.update_xaxes(tickformat="%m.%Y")  # 01.2025 вместо Jan 2025

app = fig.data[2]
print(f"{type(fig).__name__}: трасс — {len(fig.data)}; у трассы «{app.name}» тип {app.type}, точек — {len(app.x)}")
print("Разделы описания графика:", list(fig.to_dict()))
print(app.hovertemplate)

(120, 4)
  month     channel   revenue   mln
2024-01 маркетплейс 7471000.0 7.471
2024-01       поиск 4374000.0 4.374
2024-01  приложение 3363000.0 3.363


Figure: трасс — 5; у трассы «приложение» тип scatter, точек — 24
Разделы описания графика: ['data', 'layout']
канал=приложение<br>месяц=%{x}<br>млн ₽=%{y}<extra></extra>


### В подсказке видны выручка канала и его доля в продажах магазина

In [3]:
MONTHS = "январь февраль март апрель май июнь июль август сентябрь октябрь ноябрь декабрь".split()
total = sales.groupby("month")["revenue"].sum()  # выручка месяца по всем 10 каналам
by_ch["month_ru"] = [f"{MONTHS[int(m[5:]) - 1]} {m[:4]}" for m in by_ch["month"]]
by_ch["share"] = by_ch["revenue"] / by_ch["month"].map(total)

fig = px.line(by_ch, x="month", y="mln", color="channel", category_orders={"channel": top5},
              custom_data=["month_ru", "share"],  # дополнительные значения для каждой точки
              labels={"month": "месяц", "mln": "млн ₽", "channel": "канал"},
              title="С мая по июль 2025 года выручка приложения почти удвоилась")
fig.update_traces(hovertemplate="<b>%{fullData.name}</b> · %{customdata[0]}<br>"
                                "%{y:.2f} млн ₽ · %{customdata[1]:.0%} выручки месяца"
                                "<extra></extra>")  # пустой extra убирает второй ярлык с именем
fig.update_layout(separators=", ")  # десятичная запятая и пробел между разрядами
fig.update_xaxes(tickformat="%m.%Y")

print(fig.data[2].hovertemplate)
print(fig.data[2].customdata[18])  # приложение, июль 2025

<b>%{fullData.name}</b> · %{customdata[0]}<br>%{y:.2f} млн ₽ · %{customdata[1]:.0%} выручки месяца<extra></extra>
['июль 2025' 0.18281999690450396]


В Chromium подсказка для приложения за июль 2025 года содержит две строки — «приложение · июль 2025» и строкой ниже «5,91 млн ₽ · 18% выручки месяца». Выручка маркетплейса за тот же месяц составила 5 715 000 ₽. При округлении до сотых миллиона в подсказке Plotly получилось 5,71, а в таблице pandas — 5,72.

Наведите курсор на июльскую точку 2025 года на линии приложения. В подсказке появятся сумма и доля приложения в выручке магазина за месяц.

In [4]:
fig.show()

### В анимации можно выбрать любой из 24 месяцев

In [5]:
fig_a = px.bar(by_ch, x="channel", y="mln", color="channel",
               animation_frame="month",           # один кадр на месяц
               category_orders={"channel": top5},
               range_y=[0, 11],                   # одна шкала на все кадры
               labels={"channel": "", "mln": "млн ₽", "month": "месяц"},
               title="Выручка каналов по месяцам")
fig_a.update_layout(showlegend=False, separators=", ")

print("Число кадров:", len(fig_a.frames), "| период:", fig_a.frames[0].name, "…", fig_a.frames[-1].name)
print("Кнопки:", [b.label for b in fig_a.layout.updatemenus[0].buttons])
print("Месяц на 18-м шаге:", fig_a.layout.sliders[0].steps[17].label)
print("Начальная выручка приложения совпадает с первым кадром:",
      list(fig_a.data[2].y) == list(fig_a.frames[0].data[2].y))
print("Диапазон оси Y без range_y:", px.bar(by_ch, x="channel", y="mln", animation_frame="month").layout.yaxis.range)

Число кадров: 24 | период: 2024-01 … 2025-12
Кнопки: ['&#9654;', '&#9724;']
Месяц на 18-м шаге: 2025-06
Начальная выручка приложения совпадает с первым кадром: True
Диапазон оси Y без range_y: None


`&#9654;` и `&#9724;` — коды символов ▶ и ◼ в HTML. В проверке без `range_y` верхняя граница шкалы за время показа изменилась с 10,8 до 11,1 млн ₽. В основном примере шкала фиксирована.

Нажмите ▶, чтобы просмотреть 24 месяца по порядку, или выберите нужный месяц ползунком.

In [6]:
fig_a.show()

### HTML с библиотекой занимает 4,6 МБ, а описание графика в JSON — около 20 КБ

In [7]:
import os
import re

fig.write_html("channels.html")                              # include_plotlyjs=True — по умолчанию
fig.write_html("channels_cdn.html", include_plotlyjs="cdn")  # библиотеку браузер возьмёт из интернета
with open("channels.json", "w", encoding="utf-8") as f:
    f.write(fig.to_json())                                   # только фигура: данные и оформление
fig.write_image("channels.png", width=900, height=480)       # kaleido рисует через Chrome
fig.write_image("channels.svg", width=900, height=480)

for name in ["channels.html", "channels_cdn.html", "channels.json", "channels.png", "channels.svg"]:
    print(f"{name:<18} {os.path.getsize(name) / 1024:>8.1f} КБ")
print(re.search(r'src="(https://cdn[^"]+)"', open("channels_cdn.html", encoding="utf-8").read()).group(1))

channels.html        4727.3 КБ
channels_cdn.html      20.9 КБ
channels.json          20.0 КБ
channels.png           71.7 КБ
channels.svg           15.4 КБ
https://cdn.plot.ly/plotly-4.1.1.min.js


Ниже показан график из файла `channels.svg`, который создаёт этот пример.

## Размер файла, подписи и просмотр на телефоне

Следующие три примера выполняются в том же сеансе Python. В примере про заголовок мы заново прочитаем данные и построим линейный график.

### HTML с миллионом точек занимает около 26 МБ без библиотеки

In [8]:
import os

import numpy as np
import plotly.express as px

rng = np.random.default_rng(83)  # учебные случайные точки: важен только их счёт
for n in [1_000, 10_000, 100_000, 1_000_000]:
    fig = px.scatter(x=rng.normal(size=n), y=rng.normal(size=n))
    fig.write_html("points.html", include_plotlyjs="cdn")  # данные в файле, библиотека по ссылке
    kb = os.path.getsize("points.html") / 1024
    print(f"{n:>9_} точек  {fig.data[0].type:<10} {kb:>8_.0f} КБ  {kb * 1024 / n:4.1f} Б/точку"
          .replace("_", " ").replace(".", ","))

    1 000 точек  scatter          35 КБ  35,6 Б/точку


   10 000 точек  scattergl       274 КБ  28,0 Б/точку


  100 000 точек  scattergl     2 669 КБ  27,3 Б/точку


1 000 000 точек  scattergl    26 577 КБ  27,2 Б/точку


При `include_plotlyjs=True` в каждый файл добавится около 4,6 МБ. В этом примере у точки две числовые координаты. Дополнительные подписи и столбцы увеличат размер файла.

### На графике по умолчанию не подписано ни одно из 120 значений выручки

In [9]:
import re

import pandas as pd
import plotly.express as px

sales = pd.read_csv("data/viz/sales_monthly.csv")
top5 = ["сайт", "маркетплейс", "приложение", "поиск", "рассылка"]
by_ch = (sales[sales["channel"].isin(top5)]
         .groupby(["month", "channel"], as_index=False)["revenue"].sum()
         .assign(mln=lambda d: d["revenue"] / 1e6))
ru = lambda v, d=1: f"{v:.{d}f}".replace(".", ",")
words = lambda path: re.findall(r">([^<>]+)</text>", open(path, encoding="utf-8").read())

fig = px.line(by_ch, x="month", y="mln", color="channel", title="Выручка по каналам, млн ₽")
fig.update_layout(separators=", ")
fig.write_image("plain.svg")  # то, что уйдёт в письмо или на слайд
hover = [ru(v, 2) for v in by_ch["mln"]]  # числа, которые видны только при наведении
print("Значений выручки в подсказках:", len(hover), "| подписано на картинке:", sum(h in words("plain.svg") for h in hover))

app = by_ch[by_ch["channel"] == "приложение"].set_index("month")["mln"]
fig.update_layout(title=f"С мая по июль 2025 года выручка приложения выросла "
                        f"с {ru(app['2025-05'])} до {ru(app['2025-07'])} млн ₽")
fig.add_annotation(x="2025-06", y=app["2025-06"], text="релиз", ay=-40)
fig.write_image("titled.svg")
print("Заголовок и отметка на графике:", [w for w in words("titled.svg") if "выручка приложения" in w or w == "релиз"])

Значений выручки в подсказках: 120 | подписано на картинке: 0


Заголовок и отметка на графике: ['С мая по июль 2025 года выручка приложения выросла с 3,0 до 5,9 млн ₽', 'релиз']


Код ищет округлённые значения выручки в текстовых элементах SVG, а затем проверяет заголовок и подпись к июньской точке.

### Выручку по месяцам можно прочитать в таблице под графиком

In [10]:
from pathlib import Path

fig.update_layout(legend=dict(orientation="h", y=-0.15), dragmode=False)
alt = (f"График показывает выручку пяти каналов с января 2024 по декабрь 2025 года. "
       f"Выручка приложения выросла с {ru(app['2025-05'])} млн ₽ в мае до {ru(app['2025-07'])} млн ₽ "
       f"в июле 2025 года. В декабре она достигла {ru(app['2025-12'])} млн ₽.")
table = by_ch.pivot(index="month", columns="channel", values="mln")[top5]
print(alt)
print(table.loc["2025-04":"2025-08"].round(2).to_string())

# Добавляем в тот же файл краткое описание и доступную без JavaScript таблицу за все 24 месяца.
chart_html = fig.to_html(full_html=False, include_plotlyjs="cdn", config={"displayModeBar": False})
table_html = table.round(2).rename_axis("Месяц").rename_axis(None, axis=1).to_html(
    float_format=lambda value: ru(value, 2))
page = ('<!doctype html><html lang="ru"><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1">'
        '<title>Выручка по каналам продаж</title><main><h1>Выручка по каналам продаж</h1>'
        f"<p>{alt}</p>{chart_html}<h2>Выручка по месяцам, млн ₽</h2>{table_html}</main></html>")
Path("mobile.html").write_text(page, encoding="utf-8")

График показывает выручку пяти каналов с января 2024 по декабрь 2025 года. Выручка приложения выросла с 3,0 млн ₽ в мае до 5,9 млн ₽ в июле 2025 года. В декабре она достигла 8,0 млн ₽.
channel  сайт  маркетплейс  приложение  поиск  рассылка
month                                                  
2025-04  8.97         6.09        3.09   3.97      2.37
2025-05  8.77         6.02        2.98   3.72      2.24
2025-06  8.07         5.73        4.49   3.96      2.29
2025-07  8.47         5.72        5.91   4.17      2.30
2025-08  8.85         6.20        6.14   4.22      2.43


17248

В Chromium в режиме эмуляции Pixel 5 вертикальный жест по графику менял масштаб. После отключения перетаскивания тот же жест прокручивал страницу. В файле `mobile.html` есть описание графика и таблица за все 24 месяца. Их можно прочитать, даже если библиотека не загрузилась.

На графике для телефона легенда расположена снизу. Вертикальным жестом можно прокрутить страницу. Откройте файл `mobile.html`, чтобы прочитать описание графика и полную таблицу с выручкой за 24 месяца.

In [11]:
fig.show(config={"displayModeBar": False})